## Notebook 2
#### Notebook for creating a CNN to train on our preprocessed data in order to later classify skin lesions as benign or malignant. 

In [2]:
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split #Module to separate data 
from sklearn.preprocessing import LabelEncoder #For converting labels to integers

#Downloads the files created in Notebook 1 
#Project folder
DATA_DIR = Path("/Users/erinekman/cb1020/Hud-cancer/preprocessed_data")

#Paths to the numpy arrays
images_path = DATA_DIR / "images.npy"
labels_path = DATA_DIR / "labels.npy"

images = np.load(images_path)
labels = np.load(labels_path)

print("Bilder shape:", images.shape)
print("Labels shape:", labels.shape)

#Separation of data, 70/15/15

#15% test data
images_rest, images_test, labels_rest, labels_test = train_test_split(
    images,
    labels,
    test_size=0.15,
    random_state=42,
    stratify=labels
)

#70 % training, 15 % validation
images_traning, images_validering, labels_traning, labels_validering = train_test_split(
    images_rest,
    labels_rest,
    test_size=0.17647,  #gives a total of 15 % validation data
    random_state=42,
    stratify=labels_rest
)

print("Träning:", images_traning.shape, labels_traning.shape)
print("Validering:", images_validering.shape, labels_validering.shape)
print("Test:", images_test.shape, labels_test.shape)

#Saving the test data 
TEST_DIR = Path("test_data")
TEST_DIR.mkdir(parents=True, exist_ok=True)

np.save(TEST_DIR / "images_test.npy", images_test)
np.save(TEST_DIR / "labels_test.npy", labels_test)
print("Testdata sparad!")

#Defining the parameters for the training
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

WEIGHTS = "imagenet"      # Weights from imagenet
POOLING = "max"           # Max pooling
TRAINABLE = False         # False = freeze trainin

DENSE_UNITS = 256
DROPOUT = 0.3

PATIENCE = 5
RESTORE_BEST_WEIGHTS = True

#DATA SETS

#Convert labels to integer values  
encoder = LabelEncoder()

labels_traning = encoder.fit_transform(labels_traning)
labels_validering = encoder.transform(labels_validering)
labels_test = encoder.transform(labels_test)

#Creating data sets
train_ds = (
    tf.data.Dataset
    .from_tensor_slices((images_traning, labels_traning))
    .shuffle(len(images_traning))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset
    .from_tensor_slices((images_validering, labels_validering))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

#Creating the model

antal_klasser = len(np.unique(labels_traning))

base_model = tf.keras.applications.ResNet50V2(
    include_top=False,
    weights=WEIGHTS,
    pooling=POOLING,
    input_shape=(224, 224, 3)
)

base_model.trainable = TRAINABLE

inputs = tf.keras.Input(shape=(224, 224, 3))

x = tf.keras.applications.resnet_v2.preprocess_input(inputs)

x = base_model(x, training=False)

x = tf.keras.layers.Dense(
    DENSE_UNITS,
    activation="relu"
)(x)

x = tf.keras.layers.Dropout(DROPOUT)(x)

outputs = tf.keras.layers.Dense(
    antal_klasser,
    activation="softmax"
)(x)

model = tf.keras.Model(inputs, outputs)

#Compile

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

#Early stopping with patience = 5

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=RESTORE_BEST_WEIGHTS
)

#Start the training

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

#Saves the model
model.save("resnet50v2_model.keras")

#Summary
model.summary()

Bilder shape: (100, 224, 224, 3)
Labels shape: (100,)
Träning: (70, 224, 224, 3) (70,)
Validering: (15, 224, 224, 3) (15,)
Test: (15, 224, 224, 3) (15,)
Testdata sparad!
Epoch 1/20


/Users/erinekman/miniconda3/envs/cb1020/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.4714 - loss: 2.5481 - val_accuracy: 0.4667 - val_loss: 1.2249
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.4286 - loss: 1.3604 - val_accuracy: 0.5333 - val_loss: 0.7975
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.5286 - loss: 0.9292 - val_accuracy: 0.4000 - val_loss: 0.8117
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.5571 - loss: 1.0360 - val_accuracy: 0.5333 - val_loss: 0.8191
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.5429 - loss: 1.1728 - val_accuracy: 0.4667 - val_loss: 0.7565
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.4857 - loss: 1.2862 - val_accuracy: 0.6000 - val_loss: 0.7690
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.6143 - loss: 0.9024 - val_accuracy: 0.4667 - val_loss: 0.8569
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.5714 - loss: 0.9808 - val_accuracy: 0.5333 - val_loss: 0.7665
Epoch 9/20
3/3 ━━━━━━━━━━━━

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50v2 (Functional)         │ (None, 2048)           │    23,564,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,139,976 (95.90 MB)

 Trainable params: 525,058 (2.00 MB)

 Non-trainable params: 23,564,800 (89.89 MB)

 Optimizer params: 1,050,118 (4.01 MB)